# Alur Sistem — data nyata di setiap tahap

**Riset Deteksi Fault 4 Sensor Kelembaban**

Notebook ini memperlihatkan **satu jalur data** dari file mentah sampai siap
ditebak: berapa barisnya, bagaimana disatukan, bagaimana gangguan disisipkan,
bagaimana dipotong, dan berapa fitur yang dihasilkan.

Analogi: empat termometer mencatat tiap 30 detik. Kita satukan catatannya,
samakan jamnya, tempelkan gangguan buatan, potong jadi blok 16,7 jam, lalu
ringkas tiap blok menjadi 60 angka.

In [ ]:
import os, io, glob, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

ASSET_BASE = "https://raw.githubusercontent.com/vousmeevoyez/public-files/main/sidang/assets"
LOCAL_DIRS = ["assets", "sidang/assets", "../sidang/assets"]

def asset_path(rel):
    for base in LOCAL_DIRS:
        p = os.path.join(base, rel)
        if os.path.exists(p):
            return p
    return None

def read_asset(rel):
    p = asset_path(rel)
    if p:
        return pd.read_csv(p)
    r = requests_get(f"{ASSET_BASE}/{rel}")
    return pd.read_csv(io.BytesIO(r))

def requests_get(url):
    import requests
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return r.content

def show_png(rel, width=900):
    p = asset_path(rel)
    if p:
        display(Image(filename=p, width=width))
    else:
        display(Image(data=requests_get(f"{ASSET_BASE}/{rel}"), width=width))

print("Aset siap dibaca dari:", "folder lokal" if asset_path("11/11_rq4_performa_ann.csv") else "GitHub")

## 1 — Data mentah: empat kanal, 30 detik

Yang dicek: jumlah baris, rentang waktu, dan apakah keempat kolom ada. Kalau
kolom tidak lengkap, program berhenti dengan pesan yang jelas.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vousmeevoyez/public-files/refs/heads/main/data_sensor.csv"
RAW_CAND = ["data_sensor.csv", "../data_sensor.csv"] + glob.glob("/kaggle/input/**/data_sensor.csv", recursive=True)

def load_raw():
    for p in RAW_CAND:
        if os.path.exists(p):
            print("Sumber: file lokal ->", p)
            return pd.read_csv(p, index_col=0)
    print("Sumber: unduh dari GitHub (67 MB, sekali saja)")
    import requests
    r = requests.get(DATA_URL, timeout=300); r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), index_col=0)

df_raw = load_raw()
cols = ["kelembaban1", "kelembaban2", "kelembaban3", "kelembaban4"]
missing = [c for c in cols if c not in df_raw.columns]
if missing:
    raise ValueError(f"Kolom {missing} tidak ada. Tersedia: {list(df_raw.columns)}")

print("Baris:", len(df_raw))
print("Rentang:", df_raw.index[0], "->", df_raw.index[-1])
display(df_raw[cols].head(3))
display(df_raw[cols].describe().round(2))
show_png("ilustrasi/ILU_empat_kanal_bersih.png", 950)

## 2 — Broker: satukan kanal, jangan rata-ratakan

Broker menyatukan empat aliran menjadi satu tabel, tetapi **kolomnya tetap
empat**. Rata-rata dilakukan pada waktu (5 menit), bukan pada sensor.

Alasannya: kalau empat sensor dirata-rata, kerusakan satu sensor tersembunyi di
balik tiga sensor sehat. Notebook `02` mengukur akibatnya: F1 0,821 (4 kanal)
vs 0,748 (dirata-rata).

In [ ]:
df = df_raw[cols].copy()
df.index = pd.to_datetime(df.index, utc=True)
df = df.sort_index()

n_dup = int(df.index.duplicated().sum())
df = df[~df.index.duplicated(keep="first")]

df5 = df.resample("5min").mean()
grid = pd.date_range(df5.index[0], df5.index[-1], freq="300s")
n_gap = int(df5.reindex(grid).isna().any(axis=1).sum())
df5 = (df5.reindex(grid)
          .interpolate(method="time", limit_direction="both")
          .fillna(df5.median(numeric_only=True)))

print(f"30 detik : {len(df):,} baris")
print(f"5 menit  : {len(df5):,} baris ({(len(df5)*300)/86400:.1f} hari)")
print(f"duplikat dibuang: {n_dup} | slot bolong ditambal: {n_gap}")

seg = df5.iloc[:600]
fig, ax = plt.subplots(figsize=(12, 3.6))
for c in cols:
    ax.plot(seg.index, seg[c], lw=0.9, label=c)
ax.set_title("Empat kanal setelah sinkronisasi waktu (potongan 2 hari pertama)")
ax.set_ylabel("kelembaban (%)"); ax.legend(ncol=4, fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 3 — Injeksi fault: menempelkan kerusakan yang ada labelnya

Data asli tidak punya label "rusak". Karena itu kerusakan disisipkan sendiri,
dengan rumus yang sama di semua notebook. Empat jenis dasar:

| Jenis | Perubahan | Skala |
|---|---|---|
| Drift | naik pelan 1 poin/hari, episode 3 hari | sangat kecil |
| Spike | loncatan sesaat tiap ±67 sampel | 0,08 × simpangan baku |
| Bias | tambah 0,08 % tetap | kecil |
| Hardware | 8 % sampel macet, 5 % hilang | jelas terlihat |

Fault hanya masuk ke subset sensor, jadi tiga sensor lain tetap bersih.

In [ ]:
rng = np.random.default_rng(42)
x = df5["kelembaban1"].iloc[:864].to_numpy(dtype=float)   # 3 hari
t = np.arange(len(x))

def inject_drift(x):
    return x + np.arange(1, len(x) + 1) * (1.0 / 288.0)      # 1 poin/hari @5 menit

def inject_spike(x):
    y = x.copy(); y[::67] += 0.08 * np.nanstd(x); return y

def inject_bias(x):
    return x + 0.08

def inject_hardware(x):
    y = x.copy(); idx = rng.integers(len(x), size=len(x)); rv = rng.random(len(x))
    y[rv < 0.08] = x[idx[rv < 0.08]]
    y[rv < 0.05] = np.nan
    return y

cases = [("Drift (selisih diperbesar di panel kecil)", inject_drift),
         ("Spike", inject_spike),
         ("Bias", inject_bias),
         ("Hardware malfunction", inject_hardware)]

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, (title, fn) in zip(axes, cases):
    y = fn(x)
    ax.plot(t, x, color="#8a8a8a", lw=1.4, label="asli")
    ax.plot(t, y, color="#c0392b", lw=1.0, label="setelah gangguan")
    if np.isnan(y).any():
        ax.plot(t, pd.Series(y).ffill().bfill(), color="#e67e22", lw=0.9, ls="--", label="setelah ditambal")
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.25); ax.legend(fontsize=7, loc="upper left")
axes[-1].set_xlabel("sampel (5 menit)")
plt.tight_layout(); plt.show()

## 4 — Segmentasi: satu sinyal dipotong jadi banyak jendela

Window **N = 200 sampel** = 1.000 menit ≈ **16,7 jam**. Geser setengah window
(stride 100) supaya tidak ada peristiwa yang jatuh tepat di batas.

Window yang porsi fault-nya ≤ 1 % dibuang: isinya praktis normal, jadi
melabelinya "fault" akan mengajari model pola yang salah.

In [ ]:
N, STRIDE = 200, 100
w = np.lib.stride_tricks.sliding_window_view(df5["kelembaban1"].to_numpy(), N)[::STRIDE]
print(f"Window per kondisi: {len(w):,} | bentuk satu window: {w[0].shape}")
print(f"Durasi satu window: {N*5/60:.1f} jam")

display(read_asset("11/11_akuntansi_window.csv"))

## 5 — Dari jendela ke 60 angka

Tiap jendela dihitung entropinya pada 15 skala (τ = 1..15). Empat sensor ×
15 skala = **60 fitur**. Identitas sensor tetap: fitur S1 tidak pernah dicampur
dengan fitur S3.

Dua cara mengukur kemiripan pola:

- **EDM-Fuzzy** — jarak garis lurus antar potongan sinyal.
- **JSD-Fuzzy** — tiap potongan diubah jadi sebaran nilai, lalu dibandingkan
  bentuk sebarannya.

Gambar di bawah dibuat oleh notebook `11`; di sini hanya ditampilkan ulang.

In [ ]:
show_png("11/11_similarity_euclidean_vs_jsd.png", 950)
show_png("11/11_grid_fault_vs_asli_S1.png", 950)

## 6 — Model: 60 angka masuk, kelas fault keluar

ANN yang dipakai kecil: 60 masukan, satu atau dua lapisan tersembunyi
(32–64 neuron), keluaran sesuai jumlah kelas. Arsitektur dipilih **di dalam**
cross-validation, bukan setelah melihat data uji.

```
60 fitur  ->  [ lapisan tersembunyi ]  ->  kelas (2 / 5 / 7 / 5 / 2)
```

Klasifier yang dipakai hanya **ANN-LM asli** (Levenberg–Marquardt dengan
Jacobian analitik), bukan `lbfgs` dan bukan pelatih lain. Notebook `11` §12
menambahkan rantai kedua: dari kelas fault, tebak **sensor mana** yang rusak.
Saat pengujian nyata, label sensor tidak tersedia.

## 7 — Uji: kenapa harus CV berkelompok

Window bertetangga tumpang-tindih 50 %. Kalau data diacak biasa, potongan yang
sama bisa masuk latih dan uji sekaligus — skor jadi terlalu bagus. Karena itu
pembagian fold **dikelompokkan per blok waktu**.

| Skenario | Yang dibedakan | Kelas |
|---|---|---|
| S1 | normal vs ada fault | 2 |
| S2 | fault tunggal (4 jenis) | 5 |
| S3 | kombinasi **2 fault** (6 kombinasi) | **7** |
| S4 | kombinasi **3 fault** (4 kombinasi) | 5 |
| S5 | kombinasi **4 fault** sekaligus | **2** |

Catatan: tugas **17 kelas** (normal + 16 kondisi fault) adalah eksperimen
terpisah di notebook `01`/`04`, bukan S5. Identifikasi "sensor mana yang
rusak" juga bukan S4 — itu RQ6 di notebook `11` §12.

## 8 — Peta balik: tahap di sini → notebook asli

| Bagian notebook ini | Notebook asli | Sel |
|---|---|---|
| 1 Data mentah | `Ilustrasi` §1–§2 | 1–7 |
| 2 Broker + sinkron | `11` §2, `02` | 6–7 |
| 3 Injeksi fault | `11` §3–§3b | 9–12 |
| 4 Segmentasi | `11` §4, `07` §5 | 13–14 |
| 5 Entropi | `11` §5–§7b | 15–24 |
| 6 Model | `11` §10 | 33–40 |
| 7 Uji | `11` §10 | 33–40 |

---
**Status:** privat · commit `27737ee` · 2026-10-26 · rujukan: `sidang/MAPPING.md`